# Detection method comparison

Runs every changepoint detector back to back in one cell, on the same cached
runs, so the numbers are directly comparable.

**No MATLAB.** Reads committed caches only. Takes about two minutes.

**The protocol matters more than the ranking.** Each method's sensitivity
parameter is calibrated on the 20 null runs, where `pd` never changes so every
alarm is false. The chosen value is the most sensitive one that stays within
the false-alarm budget. Only then is latency measured, on runs that had no part
in calibration.

Latency cannot be compared without this. Any detector reaches zero latency by
firing immediately on every run; what stops it is the false-alarm budget, so
the budget has to be equal before the latencies mean anything.

The floor is **0.50 s**. The first window containing only post-switch data is
centred half a window after the switch, so no causal detector on this series
can beat that.

In [ ]:
import sys, warnings
sys.path.insert(0, "src")
warnings.filterwarnings("ignore")

import numpy as np
import detect_alt as da
import pd_switch as ps
import recheck_stim as rs

# ---------------------------------------------------------------- data
def load(name, keep=lambda s: True):
    r = rs.load_cached(name)
    if not r:
        raise SystemExit(f"no cache for {name} - check cache/ is populated")
    return rs.pair_average({s: v for s, v in r.items() if keep(s)})

NULLS = [load("runs_null_nostim"), load("runs_null_stim180_500")]
N_NULL = sum(len(s) for s in NULLS)

EVAL = {
    "unstimulated, held out": load("runs_switch", lambda s: s > 10),
    "180 Hz / 500 uA, up":    load("runs_dir_0.3to0.7_stim180_500"),
    "180 Hz / 500 uA, down":  load("runs_dir_0.7to0.3_stim180_500"),
}

# ------------------------------------------------------------- methods
# Each grid runs most-sensitive first, so calibration takes the first value
# that meets the budget.
METHODS = dict(da.METHODS)
METHODS["Max-split t (ours)"] = (ps.detect_cp2,    "thresh",  np.arange(1.0, 20.01, 0.25))
METHODS["k-means (ours)"]     = (ps.detect_kmeans, "min_gap", np.arange(0.01, 1.001, 0.01))

def fired(series, fn, **kw):
    """Alarms on a run with no switch: both 'ok' and 'false' are false alarms."""
    r = ps.evaluate(series, fn, **kw)
    return r["ok"] + len(r["false"])

def calibrate(fn, param, grid, budget):
    for x in grid:
        n = sum(fired(s, fn, **{param: float(x)}) for s in NULLS)
        if n <= budget:
            return float(x), n
    return None, None

# --------------------------------------------------- main run, budget 1
BUDGET = 1
print(f"CALIBRATION   {N_NULL} null runs, budget {BUDGET} false alarm(s)")
print("-" * 70)
def best_selectivity(fn, param, grid):
    """Lowest false-alarm count reachable while the detector can still fire.

    A method that meets the budget only by going silent has not been
    calibrated, it has been switched off. This finds the floor that matters:
    the fewest false alarms it achieves at any setting where it still detects
    real switches."""
    probe = EVAL["unstimulated, held out"]
    best = None
    for x in grid:
        if ps.evaluate(probe, fn, **{param: float(x)})["ok"] == 0:
            continue
        n = sum(fired(s, fn, **{param: float(x)}) for s in NULLS)
        best = n if best is None else min(best, n)
    return best

chosen, degenerate = {}, {}
for name, (fn, param, grid) in METHODS.items():
    x, n = calibrate(fn, param, grid, BUDGET)
    if x is None:
        degenerate[name] = "no setting meets the budget at all"
        print(f"  {name:<20} no setting meets the budget")
        continue
    if ps.evaluate(EVAL["unstimulated, held out"], fn, **{param: float(x)})["ok"] == 0:
        floor = best_selectivity(fn, param, grid)
        degenerate[name] = (f"meets the budget only by never firing; best it "
                            f"manages while still detecting is {floor}/{N_NULL}")
        print(f"  {name:<20} {param:<8} = {x:<8.3g}  SILENT at this setting - "
              f"best false-alarm rate while still firing is {floor}/{N_NULL}")
        continue
    chosen[name] = (fn, param, x)
    print(f"  {name:<20} {param:<8} = {x:<8.3g}  false alarms {n}/{N_NULL}")

if degenerate:
    print("\n  Excluded from the latency tables below, with the reason:")
    for k, v in degenerate.items():
        print(f"    {k}: {v}")
    print("  These are properties of the method on an 18-point series, not")
    print("  failures to detect. Both separate switches from noise too poorly")
    print("  to reach the budget without being turned off entirely.")

for label, series in EVAL.items():
    print(f"\n{label.upper()}   n = {len(series)}, switch at 5.0 s")
    print("-" * 70)
    print(f"  {'method':<20}{'caught':>9}{'latency':>10}{'sem':>8}{'early':>7}")
    rows = []
    for name, (fn, param, x) in chosen.items():
        r = ps.evaluate(series, fn, **{param: x})
        rows.append((r["latency"] if r["ok"] else np.inf, name, r))
    for _, name, r in sorted(rows):
        lat = f"{r['latency']:.2f}s" if r["ok"] else "--"
        sem = f"{r['sem']:.2f}" if r["ok"] > 1 and np.isfinite(r["sem"]) else "--"
        print(f"  {name:<20}{r['ok']:>6}/{r['n']}{lat:>10}{sem:>8}{len(r['false']):>7}")

# ------------------------------------------- robustness to the budget
# 20 null runs is a small calibration set, so a ranking that only holds at
# one budget is a ranking of the calibration noise.
print(f"\n\nSAME COMPARISON AT OTHER BUDGETS   caught / latency")
print("=" * 70)
for budget in (0, 1, 2):
    print(f"\nbudget {budget}/{N_NULL}")
    print(f"  {'method':<20}" + "".join(f"{k.split(',')[0][:16]:>20}" for k in EVAL))
    for name, (fn, param, grid) in METHODS.items():
        x, _ = calibrate(fn, param, grid, budget)
        if x is None:
            print(f"  {name:<20}{'(uncalibratable)':>20}")
            continue
        cells = []
        for s in EVAL.values():
            r = ps.evaluate(s, fn, **{param: x})
            cells.append(f"{r['ok']}/{r['n']} {r['latency']:.2f}s" if r["ok"]
                         else f"{r['ok']}/{r['n']}   --")
        print(f"  {name:<20}" + "".join(f"{c:>20}" for c in cells))

print("\n" + "=" * 70)
print("Latency is measured from the switch at 5.0 s; the floor is 0.50 s.")
print("'early' counts alarms before the switch, which are false alarms, not")
print("fast detections.")